## 피벗테이블과 다중 집계 함수

- `pivot_table()`로 두 개의 범주형 변수(지역x카테고리)를 축으로 하는 교차표
- 하나의 피벗테이블 안에서 평균/합계/개수 등 여러 집계 함수를 동시에 적용

In [1]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd

load_dotenv()

USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")

In [2]:
with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
    df = pd.read_sql("SELECT * FROM DELIVERY_ORDERS", conn)

df.head()

C:\Users\soldesk\AppData\Local\Temp\ipykernel_5344\504583233.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM DELIVERY_ORDERS", conn)


,ORDER_ID,MENU_NAME,CATEGORY,PRICE,RATING,REGION,ORDER_DATE
0,1,떡볶이,분식,9000,4.5,강남구,2026-08-01
1,2,치킨,치킨,22000,4.8,서초구,2026-08-02
2,3,피자,피자,25000,4.2,강남구,2026-08-03
3,4,짜장면,중식,8000,4.0,송파구,2026-08-04
4,5,마라탕,중식,13000,4.6,강남구,2026-08-05


In [3]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rc('font', family='Malgun Gothic')
mpl.rc('axes', unicode_minus=False)

In [4]:
import numpy as np

pivot_price = pd.pivot_table(
    df,
    values="PRICE",
    index="REGION",
    columns="CATEGORY",
    aggfunc="mean",
    fill_value=0,
    margins=True, # 총계 행/열 자동으로 추가
    margins_name="전체평균", # 원래 All 인데 전체평균
)
print(pivot_price.round(0))

CATEGORY      분식       일식       중식       치킨       피자     전체평균
REGION                                                       
강남구       7500.0  18250.0  11625.0  22250.0  24833.0  16036.0
서초구       7333.0  16000.0  13500.0  22000.0  26250.0  15750.0
송파구       9000.0  11000.0  11125.0  22667.0  27000.0  15125.0
전체평균      7643.0  14688.0  12083.0  22357.0  25667.0  15662.0


In [5]:
pivot_multi = pd.pivot_table(
    df,
    values="PRICE",
    index="REGION",
    columns="CATEGORY",
    aggfunc=["mean", "count"], # 평균과 건수를 동시에 분석
    fill_value=0,
)
print(pivot_multi)

                 mean                                               count     \
CATEGORY           분식       일식       중식            치킨            피자    분식 일식   
REGION                                                                         
강남구       7500.000000  18250.0  11625.0  22250.000000  24833.333333     3  2   
서초구       7333.333333  16000.0  13500.0  22000.000000  26250.000000     3  3   
송파구       9000.000000  11000.0  11125.0  22666.666667  27000.000000     1  3   

                   
CATEGORY 중식 치킨 피자  
REGION             
강남구       4  2  3  
서초구       4  2  2  
송파구       4  3  1  


In [6]:
import seaborn as sns

plt.figure(figsize=(8, 5))
# annot=True: 칸안에 숫자 표기
# 전체평균 삭제하여 큰 값에 대한 무조적인 대표값으로 취급하는 왜곡현상을 막는다.

sns.heatmap(pivot_price.drop("전체평균", axis=0).drop("전체평균", axis=1),
            annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("지역 x 카테고리 평균 가격 히트맵")
plt.show()

KeyboardInterrupt: 